# 03 — Endpoint/topology decomposition

Analysis 6.3, main Figure 4, and Appendix Figure A1. The factorial controls separate pointwise endpoint supervision from interface-free $H_0$ supervision. The notebook renders two standalone artifacts only: a quantitative absolute death-time residual map and a separate qualitative MST illustration.


In [ ]:
# 1. Cấu hình
from pathlib import Path
REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True
PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_topology_{PAIR}_v1"
SEEDS = [42, 43, 44]
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
LAMBDA_H0 = 0.75
PROBE_EVERY, PROBE_SIZE = 250, 1024
MST_ROWS = 96
RESIDUAL_BATCH_SIZE, RESIDUAL_BATCHES = 64, 16
EXECUTE = False
RENDER_MST = True
CUDA_VISIBLE_DEVICES = "0"


In [ ]:
# 2. Clone/fetch repo, dependencies, imports và output
import gc, shlex, subprocess, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if AUTO_PULL_REPO:
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        print("[git] Bỏ qua pull vì repo có tracked changes.")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
from _analysis_common import PAIRS, collect_jobs, final_checkpoint, geoode_command, load_teacher_cache, read_jsonl, run_jobs, set_paper_style, teacher_cache_path
from src import structural_audit as audit
from src.criterions.h0_topological_loss import h0_death_times, pairwise_distance
PAIR_CONFIG = PAIRS[PAIR]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
CACHE_DIR = PROJECT_DIR / "runs" / "teacher_cache"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
set_paper_style()
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Output: {RUN_ROOT}")


In [ ]:
# 3. No-teacher / endpoint / H0 / combined factorial + H0 source control
ARMS = {
    "no_teacher": ["--lambda_end", "0", "--lambda_topo", "0"],
    "endpoint_only": ["--lambda_end", "1", "--lambda_topo", "0"],
    "h0_only": ["--lambda_end", "0", "--lambda_topo", str(LAMBDA_H0), "--topo_teacher_source", "original"],
    "combined_original": ["--lambda_end", "1", "--lambda_topo", str(LAMBDA_H0), "--topo_teacher_source", "original"],
    "combined_projected": ["--lambda_end", "1", "--lambda_topo", str(LAMBDA_H0), "--topo_teacher_source", "projected"],
}
jobs = []
for arm, arm_args in ARMS.items():
    for seed in SEEDS:
        run_dir = RUN_ROOT / arm / f"seed_{seed}"
        uses_endpoint = arm not in {"no_teacher", "h0_only"}
        gauge_args = (["--gauge_align", "--gauge_rotation", "procrustes", "--gauge_refit_every", "1"] if uses_endpoint else ["--no-gauge_align", "--gauge_refit_every", "0"])
        extra = ["--projection_type", "pca", *gauge_args, "--lambda_ctr", "0", "--lambda_h1", "0", "--topo_metric", "chord", "--probe_every", str(PROBE_EVERY), "--probe_size", str(PROBE_SIZE), "--no_eval_retrieval", *arm_args]
        jobs.append({"name": f"{arm}/seed_{seed}", "arm": arm, "seed": seed, "run_dir": run_dir, "command": geoode_command(PROJECT_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA, cache_dir=CACHE_DIR, run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LR, extra=extra)})
print(f"Plan: {len(jobs)} jobs")
for job in jobs:
    print(shlex.join(job["command"]))
if EXECUTE:
    display(run_jobs(PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES))


In [ ]:
# 4. Aggregate downstream và fixed-probe structure
results = collect_jobs(jobs)
probe_rows = []
for job in jobs:
    probe = read_jsonl(Path(job["run_dir"]) / "probe_metrics.jsonl")
    if not probe.empty:
        final = probe.sort_values("global_step").iloc[-1].to_dict()
        probe_rows.append({"arm": job["arm"], "seed": job["seed"], **final})
probe = pd.DataFrame(probe_rows)
results.to_csv(RUN_ROOT / "topology_results.csv", index=False)
probe.to_csv(RUN_ROOT / "topology_probe_final.csv", index=False)
done = results.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet.")
else:
    display(done.groupby("arm")["avg_all"].agg(["mean", "std", "count"]).style.format(precision=4))
    fig, ax = plt.subplots(figsize=(8, 3.6))
    order = done.groupby("arm")["avg_all"].mean().sort_values().index
    stats = done.groupby("arm")["avg_all"].agg(["mean", "std"]).loc[order] * 100
    ax.barh(order, stats["mean"], xerr=stats["std"].fillna(0), capsize=3, color="#2A78D6")
    ax.set(xlabel="final AVG ×100", title="Endpoint/topology decomposition")
    fig.tight_layout(); fig.savefig(RUN_ROOT / "topology_decomposition.png", bbox_inches="tight"); plt.show()


In [ ]:
# 5. Standalone topology figures: qualitative MSTs and quantitative residual maps
if RENDER_MST:
    from scipy.sparse.csgraph import minimum_spanning_tree
    from transformers import AutoTokenizer
    frame = pd.read_csv(TRAIN_DATA)
    text_col = "text" if "text" in frame else "premise"
    n_rows = min(len(frame), RESIDUAL_BATCH_SIZE * RESIDUAL_BATCHES)
    n_rows = (n_rows // RESIDUAL_BATCH_SIZE) * RESIDUAL_BATCH_SIZE
    if n_rows < max(MST_ROWS, RESIDUAL_BATCH_SIZE): raise ValueError("Not enough rows for the topology probe.")
    rng = np.random.default_rng(0)
    indices = np.sort(rng.choice(len(frame), size=n_rows, replace=False))
    texts = frame.iloc[indices][text_col].astype(str).tolist()
    teacher_cache, _ = load_teacher_cache(teacher_cache_path(PROJECT_DIR, CACHE_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA))
    teacher = teacher_cache[torch.as_tensor(indices)].float()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(PAIR_CONFIG["student"])
    clouds = {"Teacher": teacher}
    for label, arm in (("Endpoint only", "endpoint_only"), (r"Endpoint + $H_0$", "combined_original")):
        model = audit.load_student(PAIR_CONFIG["student"], final_checkpoint(RUN_ROOT / arm / f"seed_{SEEDS[0]}", EPOCHS), device=device)
        clouds[label] = audit.encode_texts(model, tokenizer, texts, device=device, pooling=PAIR_CONFIG["student_pooling"], batch_size=64)["final"].float()
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Appendix Figure A1: fixed display layout; edge sets come from native spaces.
    teacher_mst = teacher[:MST_ROWS]
    centered = audit.unit(teacher_mst) - audit.unit(teacher_mst).mean(0)
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    xy = (centered @ vh[:2].T).numpy()
    def mst_edges(cloud):
        dist = pairwise_distance(audit.unit(cloud[:MST_ROWS]), metric="chord").numpy()
        shifted = dist + 1.0; np.fill_diagonal(shifted, 0.0)
        tree = minimum_spanning_tree(shifted).tocoo()
        return list(zip(tree.row, tree.col))
    fig, axes = plt.subplots(1, 3, figsize=(5.5, 1.85))
    colors = ["#6B46C1", "#DD6B20", "#2F855A"]
    for ax, ((name, cloud), color) in zip(axes, zip(clouds.items(), colors)):
        ax.plot(xy[:, 0], xy[:, 1], "o", ms=2.2, color="#1F2937", markeredgewidth=0, zorder=2)
        for i, j in mst_edges(cloud): ax.plot(xy[[i, j], 0], xy[[i, j], 1], color=color, lw=.7, alpha=.8)
        ax.set_title(name); ax.set_axis_off()
    fig.text(.5, .015, "Fixed display layout; edges are recomputed in each native space and are not identity-matched.", ha="center", fontsize=6.5, color="#6B7280")
    fig.tight_layout(rect=(0, .07, 1, 1))
    fig.savefig(RUN_ROOT / "figure_A1_h0_mst.pdf", bbox_inches="tight")
    fig.savefig(RUN_ROOT / "figure_A1_h0_mst.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Main Figure 4: absolute residuals on identical held-out mini-batches.
    def residual_map(student):
        rows = []
        for start in range(0, n_rows, RESIDUAL_BATCH_SIZE):
            stop = start + RESIDUAL_BATCH_SIZE
            teacher_deaths = h0_death_times(teacher[start:stop], metric="chord").cpu().numpy()
            student_deaths = h0_death_times(student[start:stop], metric="chord").cpu().numpy()
            rows.append(np.abs(student_deaths - teacher_deaths))
        return np.stack(rows)
    residuals = {name: residual_map(cloud) for name, cloud in clouds.items() if name != "Teacher"}
    np.savez(RUN_ROOT / "h0_absolute_residuals.npz", **{name.replace(" ", "_").replace("$", ""): value for name, value in residuals.items()})
    vmax = np.quantile(residuals["Endpoint only"], .99)
    fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.25), sharex=True, sharey=True)
    image = None
    for ax, (name, values) in zip(axes, residuals.items()):
        image = ax.imshow(values, aspect="auto", cmap="Reds", vmin=0, vmax=vmax, interpolation="nearest", rasterized=True)
        ax.set(title=name, xlabel="Sorted $H_0$ death rank")
        ax.text(.97, .95, f"median |residual| = {np.median(values):.3f}", transform=ax.transAxes, ha="right", va="top", fontsize=7, color="#7F1D1D")
    axes[0].set_ylabel("Fixed evaluation mini-batch")
    fig.colorbar(image, ax=axes, label=r"$|\delta^S-\delta^T|$", fraction=.035, pad=.03)
    fig.subplots_adjust(left=.11, right=.90, bottom=.20, top=.86, wspace=.10)
    fig.savefig(RUN_ROOT / "figure_4_h0_residual.pdf", bbox_inches="tight")
    fig.savefig(RUN_ROOT / "figure_4_h0_residual.png", dpi=300, bbox_inches="tight")
    plt.show()
